# Job Search API Audit

This notebook runs the local audit script and loads the saved JSON results.

Checks covered:
- stored jobs availability
- `/health`, `/jobs`, `/jobs/search`, `/jobs/grouped-by-currency`, `/jobs/rss`
- pagination guard behavior
- live scraper checks for `remoteok` and `remotive`


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'job_api_audit.py'
RESULT_PATH = REPO_ROOT / 'notebooks' / 'output' / 'job_api_audit_results.json'

completed = subprocess.run([sys.executable, str(SCRIPT_PATH)], cwd=str(REPO_ROOT), capture_output=True, text=True)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(f'Audit script failed with exit code {completed.returncode}')


In [ ]:
results = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
print('Generated at:', results['generated_at'])
print('Stored jobs:', results['stored_jobs_count'])
print('\nSummary:')
for item in results.get('summary', []):
    print('-', item)


In [ ]:
endpoint_checks = results['endpoint_checks']
for name, payload in endpoint_checks.items():
    print(f'\n[{name}] status={payload.get("status_code")} ok={payload.get("ok")}')
    if payload.get('json_ok'):
        body = payload.get('body', {})
        if isinstance(body, dict):
            print('count =', body.get('count'))
            print('error =', body.get('error'))
    else:
        print(payload.get('text_preview', '')[:250])

print('\nScraper checks:')
for check in results.get('scraper_checks', []):
    print('\nSources:', check.get('label'))
    print('ok:', check.get('ok'))
    print('count:', check.get('count'))
    if check.get('sample_titles'):
        print('sample titles:')
        for title in check['sample_titles'][:3]:
            print('  -', title)
    if check.get('error'):
        print('error:', check['error'])
